In [13]:
import pandas as pd
from dateutil import parser
import pytz
import difflib

In [ ]:
cfbd = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfbd_merged.csv")
box = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv")

def clean_team_name(name):
    if pd.isna(name):
        return None
    name = str(name).lower().strip()

    replacements = {
        "&": "and",
        "st.": "state",
        "st": "state ",
        "university": "",
        "univ": "",
        "u ": "",
        "-": " ",
        ".":""
    }

    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return " ".join(name.split())

for df, cols in [(cfbd, ["homeTeam", "awayTeam"]), (box, ["home", "away"])]:
    df["home_clean"] = df[cols[0]].map(clean_team_name)
    df["away_clean"] = df[cols[0]].map(clean_team_name)


def parse_et_to_utc(dt_string):
    if pd.isna(dt_string):
        return None
    local = parser.parse(dt_string)
    eastern = pytz.timezone("US/Eastern")
    local = eastern.localize(local)
    return local.astimezone(pytz.utc)

cfbd["game_datetime_utc"] = pd.to_datetime(cfbd["startDate"], utc=True)
box["game_datetime_utc"] = box["date"].map(parse_et_to_utc)

for df in [cfbd, box]:
    df["game_datetime_round"] = df["game_datetime_utc"].dt.round("10min")

def fuzzy_match_name(name,choices, threshold=0.8):
    if pd.isna(name):
        return None
    matches = difflib.get_close_matches(name, choices, n=1, cutoff=threshold)
    return matches[0] if matches else None

cfbd_names = list(cfbd["home_clean"].dropna().unique())

box["home_clean_fuzzy"] = box["home_clean"].map(lambda x: fuzzy_match_name(x, cfbd_names))
box["away_clean_fuzzy"] = box["away_clean"].map(lambda x: fuzzy_match_name(x, cfbd_names))


merged = cfbd.merge(
    box,
    how="inner",
    left_on=["game_datetime_round", "home_clean", "away_clean"],
    right_on=["game_datetime_round", "home_clean_fuzzy", "away_clean_fuzzy"],
    suffixes=("_cfbd","_box")
)

print("Merged sample:")
display(merged.head())

print(f"Merged rows: {len(merged)}")

/var/folders/5l/415lzmwx6x99xzsk4hxsyy980000gp/T/ipykernel_37076/2745145509.py:1: DtypeWarning: Columns (31,40) have mixed types. Specify dtype option on import or set low_memory=False.
  cfbd = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfbd_merged.csv")


Merged sample:


,id_x,season_cfbd,week_cfbd,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance_cfbd,...,pen_yards_home,possession_away,possession_home,attendance_box,tv,home_clean_box,away_clean_box,game_datetime_utc_box,home_clean_fuzzy,away_clean_fuzzy
0,63843,2006,3,regular,2006-09-16T04:00:00.000Z,False,True,False,False,NaN,...,92.0,28.07,31.93,32008.0,NaN,hawaii,hawaii,2006-09-16 04:00:00+00:00,hawai'i,hawai'i
1,63845,2006,6,regular,2006-10-07T04:00:00.000Z,False,True,False,True,NaN,...,109.0,26.03,33.97,33761.0,NaN,hawaii,hawaii,2006-10-07 04:00:00+00:00,hawai'i,hawai'i
2,63846,2006,9,regular,2006-10-28T04:00:00.000Z,False,True,False,True,NaN,...,38.0,29.62,30.83,34051.0,NaN,hawaii,hawaii,2006-10-28 04:00:00+00:00,hawai'i,hawai'i
3,401643703,2024,1,regular,2024-08-31T04:00:00.000Z,True,True,False,False,17037.0,...,105.0,34.37,25.63,17037.0,MWN,utah state ate,utah state ate,2024-08-31 04:00:00+00:00,utah state ate,utah state ate


Merged rows: 4


In [19]:
len(merged)

4